# Model 4: CNN with Batch Normalization

## Architecture
- **3 Convolutional Layers**: 32 → 64 → 128 filters
- **Batch Normalization**: After each conv layer
- **Max Pooling**: After each conv layer (2x2)
- **2 Fully Connected Layers**: 256 → 10 classes
- **No Dropout** (BatchNorm has some regularizing effect)

In [ ]:
import sys
import os

current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

src_path = os.path.join(current_dir, 'src')
if os.path.exists(src_path):
    sys.path.append(src_path)
    print(f"✅ Found src at: {src_path}")
else:
    alt_path = '/content/cnn-image-classification/src'
    if os.path.exists(alt_path):
        sys.path.append(alt_path)
        print(f"✅ Found src at: {alt_path}")
    else:
        print(f"❌ src folder not found!")
        print("Files in current directory:", os.listdir('.'))

In [ ]:
import torch
import torch.nn as nn
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from data_loader import AnimalsDataLoader
from models import Model4_BatchNorm
from train import ModelTrainer

print("✅ All imports successful!")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

MODEL_NAME = 'model_04_batchnorm'
NUM_CLASSES = 10
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.001

print(f"\nModel: {MODEL_NAME}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")

In [ ]:
print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

data_loader = AnimalsDataLoader(
    data_dir='data/animals',
    batch_size=BATCH_SIZE,
    image_size=224,
    use_english_names=True
)

train_loader, val_loader, test_loader = data_loader.load_data(
    val_ratio=0.15,
    test_ratio=0.15
)

print("\n✓ Data loaded successfully!")
print(f"  Training: {len(train_loader.dataset)} images ({len(train_loader)} batches)")
print(f"  Validation: {len(val_loader.dataset)} images ({len(val_loader)} batches)")
print(f"  Test: {len(test_loader.dataset)} images ({len(test_loader)} batches)")

In [ ]:
print("\n" + "="*60)
print("CREATING MODEL")
print("="*60)

model = Model4_BatchNorm(num_classes=NUM_CLASSES)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"\n{model}")

In [ ]:
print("\n" + "="*60)
print("INITIALIZING TRAINER")
print("="*60)

trainer = ModelTrainer(
    model=model,
    device=device,
    model_name=MODEL_NAME,
    save_dir='models_pickles'
)

print(f"✓ Trainer initialized")
print(f"  Save directory: {trainer.save_dir}")

In [ ]:
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.0,
    patience=5
)

print(f"\n✅ Training Complete!")
print(f"  Best Validation Accuracy: {trainer.best_val_acc:.2f}%")
print(f"  Best Epoch: {trainer.best_epoch}")

In [ ]:
print("\n" + "="*60)
print("TRAINING VISUALIZATION")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(trainer.history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(trainer.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].axvline(x=trainer.best_epoch-1, color='green', linestyle='--', alpha=0.5, label=f'Best Epoch ({trainer.best_epoch})')
axes[0].set_title(f'Loss During Training - {MODEL_NAME}', fontsize=14)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(trainer.history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(trainer.history['val_acc'], label='Val Acc', linewidth=2)
axes[1].axvline(x=trainer.best_epoch-1, color='green', linestyle='--', alpha=0.5, label=f'Best Epoch ({trainer.best_epoch})')
axes[1].set_title(f'Accuracy During Training - {MODEL_NAME}', fontsize=14)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
os.makedirs('outputs', exist_ok=True)
plt.savefig(f'outputs/{MODEL_NAME}_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Training history saved to: outputs/{MODEL_NAME}_history.png")

In [ ]:
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

test_results = trainer.evaluate(test_loader)

print(f"\nTest Results:")
print(f"  Accuracy: {test_results['accuracy']:.2f}%")
print(f"  Total samples tested: {len(test_results['true_labels'])}")

In [ ]:
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

results = {
    'model_name': MODEL_NAME,
    'model_class': model.__class__.__name__,
    'test_accuracy': test_results['accuracy'],
    'best_val_acc': trainer.best_val_acc,
    'best_epoch': trainer.best_epoch,
    'total_epochs': len(trainer.history['train_loss']),
    'history': trainer.history,
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs': EPOCHS,
        'num_classes': NUM_CLASSES,
        'total_params': total_params,
        'trainable_params': trainable_params,
    },
    'device': str(device)
}

# Save as JSON
json_path = f'models_pickles/{MODEL_NAME}_results.json'
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to: {json_path}")

# Verify pickle was saved
pickle_path = f'models_pickles/{MODEL_NAME}.pkl'
if os.path.exists(pickle_path):
    size_mb = os.path.getsize(pickle_path) / (1024 * 1024)
    print(f"✓ Model pickle saved: {pickle_path} ({size_mb:.2f} MB)")
else:
    print("❌ Model pickle not found!")

## Observations

### What I Notice
1. **Training Accuracy**: 94.36%
2. **Validation Accuracy**: 61.47%
3. **Test Accuracy**: 62.16%
4. **Overfitting Gap**: 32.89%

### Key Takeaways
- The model is showing same overfitting gap overall performance as the dropout50 model
- Early stopping triggered after after 12 epochs